# 🌍 Composite Climate Risk Score Engine
### Multi-Hazard Risk Aggregation for Indian Real Estate

This notebook combines all five individual hazard scores (drought, flood, heatwave,
cyclone, landslide) into a single **composite climate risk score** per city per year.

**Weights:**
| Hazard | Weight |
|--------|--------|
| Drought | 0.25 |
| Flood | 0.30 |
| Heatwave | 0.20 |
| Cyclone | 0.15 |
| Landslide | 0.10 |

---
## Step 1 — Load All Individual Risk Score CSVs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = "../data/outputs"

# ----- Load each module's risk scores -----
drought   = pd.read_csv(os.path.join(OUTPUT_DIR, "drought_risk_scores.csv"))
flood     = pd.read_csv(os.path.join(OUTPUT_DIR, "flood_risk_scores.csv"))
heatwave  = pd.read_csv(os.path.join(OUTPUT_DIR, "heatwave_risk_scores.csv"))
cyclone   = pd.read_csv(os.path.join(OUTPUT_DIR, "cyclone_risk_scores.csv"))
landslide = pd.read_csv(os.path.join(OUTPUT_DIR, "landslide_risk_scores.csv"))

# ----- Standardize column names for merging -----
# Each module has its own score column name — rename to a common pattern
drought_slim   = drought[["city", "year", "drought_risk_score"]].copy()
flood_slim     = flood[["city", "year", "flood_risk_score"]].copy()
heatwave_slim  = heatwave[["city", "year", "heatwave_risk_score"]].copy()
cyclone_slim   = cyclone[["city", "year", "cyclone_risk_score"]].copy()
landslide_slim = landslide[["city", "year", "landslide_risk_score"]].copy()

print(f"Drought:   {len(drought_slim)} rows, {drought_slim['city'].nunique()} cities")
print(f"Flood:     {len(flood_slim)} rows, {flood_slim['city'].nunique()} cities")
print(f"Heatwave:  {len(heatwave_slim)} rows, {heatwave_slim['city'].nunique()} cities")
print(f"Cyclone:   {len(cyclone_slim)} rows, {cyclone_slim['city'].nunique()} cities")
print(f"Landslide: {len(landslide_slim)} rows, {landslide_slim['city'].nunique()} cities")

---
## Step 2 — Merge All Scores on (city, year)

Not all cities appear in all modules (e.g., Shimla is only in landslide, not cyclone).
We use **outer joins** and fill missing scores with 0 (no risk for non-applicable hazards).

In [ ]:
# ----- Merge all dataframes on (city, year) using outer joins -----
composite = drought_slim.copy()

for df in [flood_slim, heatwave_slim, cyclone_slim, landslide_slim]:
    composite = composite.merge(df, on=["city", "year"], how="outer")

# ----- Fill missing scores with 0 (hazard not applicable to that city) -----
score_cols = [
    "drought_risk_score", "flood_risk_score", "heatwave_risk_score",
    "cyclone_risk_score", "landslide_risk_score"
]
composite[score_cols] = composite[score_cols].fillna(0)

print(f"Composite dataframe: {len(composite)} rows, {composite['city'].nunique()} unique cities")
print(f"\nCities: {sorted(composite['city'].unique())}")
print(f"Years:  {sorted(composite['year'].unique())}")

---
## Step 3 — Compute Weighted Composite Score

In [ ]:
# ----- Weights (sum to 1.0) -----
WEIGHTS = {
    "drought_risk_score":   0.25,
    "flood_risk_score":     0.30,
    "heatwave_risk_score":  0.20,
    "cyclone_risk_score":   0.15,
    "landslide_risk_score": 0.10,
}

# ----- Compute weighted average -----
composite["composite_score"] = sum(
    composite[col] * weight for col, weight in WEIGHTS.items()
).round(2)

# ----- Assign composite category -----
def composite_category(score):
    if score <= 25:
        return "LOW"
    elif score <= 50:
        return "MEDIUM"
    elif score <= 75:
        return "HIGH"
    else:
        return "VERY HIGH"

composite["composite_category"] = composite["composite_score"].apply(composite_category)

# ----- Sort and display -----
composite = composite.sort_values(["city", "year"]).reset_index(drop=True)
print(composite.to_string(index=False))

---
## Step 4 — Heatmap Visualization

Cities as rows, hazards as columns, colour intensity = risk score.
Shows the **latest year** (2023) or most recent available year per city.

In [ ]:
# ----- Get latest year per city for the heatmap -----
latest = composite.groupby("city").apply(
    lambda g: g.loc[g["year"].idxmax()]
).reset_index(drop=True)

# ----- Prepare heatmap data -----
hazard_cols = [
    "drought_risk_score", "flood_risk_score", "heatwave_risk_score",
    "cyclone_risk_score", "landslide_risk_score", "composite_score"
]
hazard_labels = ["Drought", "Flood", "Heatwave", "Cyclone", "Landslide", "COMPOSITE"]

# Sort cities by composite score (highest risk at top)
latest = latest.sort_values("composite_score", ascending=True)

heatmap_data = latest[hazard_cols].values
city_labels  = latest["city"].values
year_labels  = latest["year"].values

# ----- Plot -----
fig, ax = plt.subplots(figsize=(12, max(8, len(city_labels) * 0.5)))

im = ax.imshow(heatmap_data, cmap="YlOrRd", aspect="auto", vmin=0, vmax=100)

# Axis labels
ax.set_xticks(range(len(hazard_labels)))
ax.set_xticklabels(hazard_labels, rotation=45, ha="right", fontsize=11)
ax.set_yticks(range(len(city_labels)))
ax.set_yticklabels([f"{c} ({y})" for c, y in zip(city_labels, year_labels)], fontsize=10)

# Add score text inside each cell
for i in range(len(city_labels)):
    for j in range(len(hazard_labels)):
        val = heatmap_data[i, j]
        color = "white" if val > 50 else "black"
        ax.text(j, i, f"{val:.0f}", ha="center", va="center", color=color, fontsize=9)

ax.set_title("Multi-Hazard Climate Risk Scores — Indian Cities (Latest Year)", fontsize=14, pad=15)
plt.colorbar(im, ax=ax, label="Risk Score (0–100)", shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "composite_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved to data/outputs/composite_heatmap.png")

---
## Step 5 — Save Composite Scores

In [ ]:
# ----- Save full composite table -----
composite_path = os.path.join(OUTPUT_DIR, "composite_risk_scores.csv")
composite.to_csv(composite_path, index=False)
print(f"✓  Saved composite risk scores → {composite_path}")
print(f"   {len(composite)} rows, {composite['city'].nunique()} cities")

# ----- Summary statistics -----
print("\n--- Risk Category Distribution ---")
print(composite["composite_category"].value_counts().to_string())

# ----- Top 10 highest-risk city-years -----
print("\n--- Top 10 Highest Composite Risk ---")
top10 = composite.nlargest(10, "composite_score")[["city", "year", "composite_score", "composite_category"]]
print(top10.to_string(index=False))